# Day 48 — Transfer learning & CNN basics
Objectives:
- Use a pretrained CNN (ResNet18) and fine-tune last layers.
- Small dataset or subset recommended.
Note: torchvision datasets may download; adjust if offline.

In [ ]:
import torch
import torchvision.transforms as T
from torchvision import models
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in model.parameters(): p.requires_grad=False
model.fc = torch.nn.Linear(model.fc.in_features, 2)
sum(p.requires_grad for p in model.parameters()), model.fc


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — offline-safe transfer learning, frozen parameters, and cautious fine-tuning

### Mental model

Transfer learning reuses a **backbone** that learned general image
features and replaces its task-specific head. Freezing sets
`requires_grad=False`; it does not automatically switch batch
normalization or dropout into evaluation behavior. The optimizer should
receive only parameters intended to change.

Pretrained weights are an external artifact with provenance, license,
preprocessing, and cache requirements. The default lesson must make a
connected first-use download explicit and provide a `weights=None`
architecture-only fallback so offline execution never surprises the
learner.

### Read the API before running it

- **`models.resnet18(weights=...)`:** constructs the architecture and optionally loads a specific versioned weight bundle.
- **`parameter.requires_grad = False`:** excludes a parameter from autograd updates but does not change module mode.
- **`model.fc = nn.Linear(model.fc.in_features, classes)`:** replaces the classifier head while preserving the backbone feature width.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — build and verify a no-download frozen backbone

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Random backbone features are a mechanics fallback, not equivalent to pretrained transfer performance.

In [ ]:
import torch
from torchvision import models

model = models.resnet18(weights=None)  # explicit offline architecture
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = torch.nn.Linear(model.fc.in_features, 3)

trainable = [name for name, p in model.named_parameters() if p.requires_grad]
print(trainable)
assert trainable == ["fc.weight", "fc.bias"]

**Expected observation:** Only the newly assigned head parameters are trainable; no pretrained asset was downloaded.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — apply the matching tensor normalization contract

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Those mean/std values are paired with the selected pretrained-weight recipe; arbitrary weights may require another contract.

In [ ]:
import torch
from torchvision.transforms import Normalize

image = torch.full((3, 4, 4), 0.5)
normalize = Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)
transformed = normalize(image)
print({"shape": tuple(transformed.shape),
       "channel_means": transformed.mean(dim=(1, 2)).tolist()})
assert transformed.shape == image.shape

**Expected observation:** Normalization preserves channel/height/width shape while applying a different affine transform to each channel.

### Debugging and practice ramp

**Common mistake:** Downloading default weights silently, freezing parameters but optimizing all of them, or using training augmentation during validation.

**Diagnostic:** Print weight enum/cache path, trainable parameter names/counts, module modes, transform pipeline, class mapping, and one batch shape.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define offline-safe transfer learning, frozen parameters, and cautious fine-tuning in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not fine-tune until the frozen-head baseline, validation transform, and offline/provenance behavior are reproducible.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Load a small image folder with `ImageFolder` and `DataLoader`.

**Verify:** For task `Load a small image folder with ImageFolder and DataLoader`, show the relevant row/group/time identities and assert the training and evaluation information boundaries are disjoint.






2. Train only the classifier head for a few epochs.

**Verify:** For task `Train only the classifier head for a few epochs`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.






3. Unfreeze the final ResNet block and fine-tune it with a lower learning rate.

**Verify:** For task `Unfreeze the final ResNet block and fine-tune it with a lower learning rate`, demonstrate the concrete requirement “3. Unfreeze the final ResNet block and fine-tune it with a lower learning rate” with explicit inputs, observable output, and one counterexample.







### Progressive hints

1. Use `data/train/<class>/...` and `data/valid/<class>/...`. Resize/crop to the
   expected input size and normalize with the selected weights' documented
   transform. Start with `num_workers=0` for portable notebooks.
2. Pass only `model.fc.parameters()` to the optimizer and record validation loss
   as well as accuracy.
3. Set `requires_grad=True` for `model.layer4`, then use parameter groups: a
   smaller rate for the backbone than for the new head.

The separate solution also demonstrates augmentation and `OneCycleLR`. Add
those only after the head-only and fine-tuning baselines are reproducible.

### Additional mastery practice

Make pretrained weights, image transforms, frozen state, and offline fallback part of one explicit transfer-learning contract.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Transform mismatch diagnosis:** Compare predictions when validation images use the training transform with random crop/flip versus a deterministic validation transform. Explain the metric instability.
   **Progressive hint:** Augmentation belongs to training. Validation should apply deterministic resize/crop and the normalization expected by the selected weights.

**Verify:** For task `Transform mismatch diagnosis: Compare predictions when validation images use the training tra...`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then state one precise claim, the evidence supporting it, the governing assumption, and a counterexample or limitation.







5. **Frozen-state edge case:** Freeze a pretrained backbone containing BatchNorm. Explain the difference between `requires_grad=False` and putting frozen modules in evaluation mode.
   **Progressive hint:** requires_grad controls parameter gradients; train/eval controls BatchNorm running statistics and Dropout behavior.

**Verify:** For task `Frozen-state edge case: Freeze a pretrained backbone containing BatchNorm. Explain the differ...`, state one precise claim, the evidence supporting it, the governing assumption, and a counterexample or limitation; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







6. **Offline fallback design:** Make the lesson runnable when pretrained weights are not cached. Detect cache availability, offer an explicit connected preload step, and provide a tiny randomly initialized CNN smoke path.
   **Progressive hint:** Never trigger an undocumented download. Report whether results use pretrained or random weights because their learning goals differ.

**Verify:** For task `Offline fallback design: Make the lesson runnable when pretrained weights are not cached. Det...`, produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Transform mismatch diagnosis


# Practice 5 — Frozen-state edge case


# Practice 6 — Offline fallback design
